# 🚀 AI Prompt Generator - Qwen 2.5 3B QLoRA Fine-Tuning Notebook

This notebook fine-tunes **Qwen 2.5 3B Instruct** on your prompt generator dataset (`train.jsonl`, `validation.jsonl`, `test.jsonl`).

### ⚡ Key Features:
- **Fast QLoRA Training** with Unsloth (4-bit quantization, 2x faster & uses 60% less VRAM).
- **Short Data Sample Mode (`SAMPLE_DATA_ONLY = True`)**: Runs a fast test training in **3-5 minutes** on free Google Colab T4 GPU.
- **Full Training Switch**: Easily toggle to full dataset training when ready (3-4 hours).
- **Ollama GGUF Export**: Converts trained model directly into `.gguf` format for local use in your Chrome extension and FastAPI backend.

In [ ]:
# 1. Install Unsloth & fine-tuning dependencies
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes datasets triton

In [ ]:
# 2. Upload dataset files (train.jsonl, validation.jsonl)
import os
from google.colab import files
from datasets import load_dataset

print("Please upload train.jsonl and validation.jsonl from your data/processed directory:")
if not os.path.exists("train.jsonl"):
    uploaded = files.upload()

# Load datasets into Hugging Face Datasets
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "validation.jsonl"
})

print(f"Loaded Raw Train split: {len(dataset['train'])} samples")
print(f"Loaded Raw Validation split: {len(dataset['validation'])} samples")

In [ ]:
# 3. Configure Sub-dataset Sampling for Quick Testing (3-5 min run)
SAMPLE_DATA_ONLY = True   # Set to False when you are ready to train on full dataset (~3-4 hrs)
MAX_TRAIN_SAMPLES = 500  # Number of samples to train on for quick demo
MAX_EVAL_SAMPLES = 50    # Number of validation samples

if SAMPLE_DATA_ONLY:
    train_data = dataset["train"].select(range(min(MAX_TRAIN_SAMPLES, len(dataset["train"]))))
    eval_data = dataset["validation"].select(range(min(MAX_EVAL_SAMPLES, len(dataset["validation"]))))
    print(f"⚡ SAMPLE MODE ACTIVE: Training on {len(train_data)} samples, Evaluating on {len(eval_data)} samples.")
else:
    train_data = dataset["train"]
    eval_data = dataset["validation"]
    print(f"🔥 FULL DATASET ACTIVE: Training on {len(train_data)} samples.")

In [ ]:
# 4. Load Base Model & Tokenizer (Qwen 2.5 3B Instruct 4-bit)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled in Google Colab! Please go to top menu: Runtime -> Change runtime type -> Select 'T4 GPU' -> Save, then restart session.")

from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True # 4-bit quantization to fit free Colab T4 GPU memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model and Tokenizer loaded successfully!")

In [ ]:
# 5. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("LoRA adapters attached to model.")

In [ ]:
# 6. Format Dataset for Qwen ChatML Structure
EOS_TOKEN = tokenizer.eos_token

def format_prompts(examples):
    instructions = examples["instruction"]
    responses = examples["response"]
    texts = []
    for inst, resp in zip(instructions, responses):
        text = f"<|im_start|>user\n{inst}<|im_end|>\n<|im_start|>assistant\n{resp}<|im_end|>{EOS_TOKEN}"
        texts.append(text)
    return { "text" : texts }

train_dataset = train_data.map(format_prompts, batched = True)
eval_dataset = eval_data.map(format_prompts, batched = True)

print("Sample Formatted Prompt:\n", train_dataset[0]["text"][:300])

In [ ]:
# 7. Configure & Launch Training with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 60 if SAMPLE_DATA_ONLY else -1,
    num_train_epochs = 1 if SAMPLE_DATA_ONLY else 3,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    eval_strategy = "steps" if SAMPLE_DATA_ONLY else "epoch",
    eval_steps = 20,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs",
)

# Version-safe initialization for SFTTrainer
try:
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = training_args,
    )
except TypeError:
    trainer = SFTTrainer(
        model = model,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        args = training_args,
    )

print("Starting Fine-Tuning...")
trainer_stats = trainer.train()
print("Training Complete!")

In [ ]:
# 8. Test Fine-Tuned Model Inference
FastLanguageModel.for_inference(model)

test_instruction = (
    "Transform the following frontend website requirement into a detailed implementation-ready prompt:\n\n"
    "Requirement: Build a modern portfolio website for a freelance UX designer with case studies, testimonials, and contact form."
)

prompt = f"<|im_start|>user\n{test_instruction}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result = tokenizer.batch_decode(outputs)
print("Generated Response:\n", result[0].split("<|im_start|>assistant\n")[-1])

In [ ]:
# 9. Save & Export Model (LoRA Adapter & GGUF for Ollama)
# Save LoRA Adapter
model.save_pretrained("prompt_generator_lora")
tokenizer.save_pretrained("prompt_generator_lora")
print("Saved LoRA adapter weights to ./prompt_generator_lora")

# Export to 4-bit GGUF format for Ollama
print("Converting and saving model to GGUF format for Ollama...")
model.save_pretrained_gguf("prompt_generator_gguf", tokenizer, quantization_method = "q4_k_m")

# Download GGUF file to your local computer
from google.colab import files
files.download("prompt_generator_gguf/unsloth.Q4_K_M.gguf")

### 🎯 How to use your trained model in Ollama & Chrome Extension

1. **Move downloaded `.gguf` file to your project directory**:
   Place the downloaded `unsloth.Q4_K_M.gguf` file inside your project root or `training/` folder.

2. **Create a `Modelfile`**:
   ```dockerfile
   FROM ./unsloth.Q4_K_M.gguf
   PARAMETER stop "<|im_end|>"
   PARAMETER temperature 0.7
   ```

3. **Import Model into Ollama**:
   ```bash
   ollama create prompt-generator -f Modelfile
   ```

4. **Verify Model in Ollama**:
   ```bash
   ollama run prompt-generator "Create a landing page for an AI startup"
   ```

5. **Connect Backend & Chrome Extension**:
   Update your `.env` file in the project root:
   ```env
   LLM_PROVIDER=ollama
   OLLAMA_BASE_URL=http://127.0.0.1:11434
   OLLAMA_MODEL=prompt-generator
   ```
   Start the FastAPI server (`uvicorn backend.main:app --reload`) and use the Chrome Extension!